## 1. Autoloader filetrigger to ingest the streaming data by enabling filenotification feature

####IAM Access for Storage Account and Resource group:
1.storage account => IAM => Add => Add role assignment => Role => Privileged administrator roles => Contributor => Member => Service principle app => Grant access

2.storage account => IAM => Add => Add role assignment => Role => Job function roles => Storage blob data Contributor => Member => Service principle app => Grant access

3.storage account => IAM => Add => Add role assignment => Role => Job function roles => Storage queue data Contributor => Member => Service principle app => Grant access

4.Resource group => IAM => Add => Add role assignment => Role => Job function roles => EventGrid EventSubscription Contributor => Member => Service principle app => Grant access



In [0]:
# 1. Define your Azure Data Lake Storage Gen2 (ADLS Gen2) paths
source_path = "abfss://telecom-cont-1@soundharadls.dfs.core.windows.net/src_dir_adls/" 
target_table = "soundhar_ws.default.cust_stream"
checkpoint_path = "abfss://telecom-cont-1@soundharadls.dfs.core.windows.net/_checkpoints/my_pipeline1/"
schema_path = "abfss://telecom-cont-1@soundharadls.dfs.core.windows.net/_schemas/my_pipeline1/"

# 2. Define the Auto Loader configuration dictionary for Azure
cloudFilesConf = {
    "cloudFiles.format": "csv",
    "cloudFiles.schemaLocation": schema_path,
    "cloudFiles.useNotifications": "true",
    "header": "true",
    
    # Azure-specific configuration required to create Event Grid and Queues:
    "cloudFiles.subscriptionId": "",
    "cloudFiles.resourceGroup": "",
    "cloudFiles.tenantId": "",
    "cloudFiles.clientId": "",
    "cloudFiles.clientSecret": ""
}

# 3. spark config
storage_account = "soundharadls"
container_name = "telecom-cont-1"
tenant_id = ""
service_principal_id =""
service_principal_secret = ""

spark.conf.set(f"fs.azure.account.auth.type.{storage_account}.dfs.core.windows.net", "OAuth") # open authorization
spark.conf.set(f"fs.azure.account.oauth.provider.type.{storage_account}.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set(f"fs.azure.account.oauth2.client.id.{storage_account}.dfs.core.windows.net", service_principal_id)
spark.conf.set(f"fs.azure.account.oauth2.client.secret.{storage_account}.dfs.core.windows.net", service_principal_secret)
spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{storage_account}.dfs.core.windows.net", f"https://login.microsoftonline.com/{tenant_id}/oauth2/token")


# 4. Read Stream
df = (spark.readStream
      .format("cloudFiles")
      .options(**cloudFilesConf)
      .load(source_path))

# 5. Write Stream
query = (df.writeStream
         .format("delta")
         .option("checkpointLocation", checkpoint_path)
         .trigger(availableNow=True)
         .toTable(target_table))



In [0]:
%sql
select count(*) from soundhar_ws.default.cust_stream;